# Esquema: clasificación binaria + sklearn + PyTorch (MVP)

CSV → target **0/1** → features → split → modelos **sklearn** → red **PyTorch** → comparación en test.

| Paso | Contenido |
|------|-----------|
| 1–5 | CSV (`data/datos_spam.csv`), codificación del target, features, split |
| 6 | Modelos **sklearn** (`build_models()`) |
| 7 | Red **PyTorch** (`TabularBinaryNet`) |
| 8 | **Comparación global** + reporte del mejor |

> Ejecuta el notebook desde `13-esquemas-sklearn-pytorch/`.


## 1. CSV

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_spam.csv")
display(df)


,Palabras_En_Texto,Etiqueta_Texto
0,3.0,no
1,8.0,no
2,12.0,no
3,25.0,si
4,30.0,si
5,45.0,si
6,5.0,no
7,NaN,si
8,22.0,si
9,7.0,no


## 2. Target → 0/1

In [2]:
MAPA_BINARIO = {"no": 0, "si": 1}
df = df.dropna(subset=["Etiqueta_Texto"]).copy()
y = df["Etiqueta_Texto"].str.strip().str.lower().map(MAPA_BINARIO).astype(int)


## 3. Feature

In [3]:
palabras = df["Palabras_En_Texto"].astype(float)
X = pd.DataFrame({"Palabras_En_Texto": palabras.fillna(palabras.median())})


## 4. Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 5–6. Modelos sklearn

In [5]:
def build_models():
    """Comenta entradas del dict para excluir modelos."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            eval_metric="logloss",
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()

filas = []
predicciones_test = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predicciones_test[nombre] = pred
    filas.append({"modelo": nombre, "accuracy": accuracy_score(y_test, pred)})
# métricas agregadas en el paso 8



## 7. Red neuronal (PyTorch)

Misma arquitectura tabular (BatchNorm + Dropout). **Una logit** de salida + `BCEWithLogitsLoss`.


In [6]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

scaler_nn = StandardScaler()
X_tr = scaler_nn.fit_transform(X_train)
X_te = scaler_nn.transform(X_test)
n_in = X_tr.shape[1]


class TabularBinaryNet(nn.Module):
    """MLP tabular; salida 1 logit (clasificación binaria)."""

    def __init__(self, n_features: int, dropout_rate: float = 0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(-1)  # (batch,)


modelo_nn = TabularBinaryNet(n_in).to(device)
criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=1e-2)

X_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_t = torch.tensor(y_train.values, dtype=torch.float32, device=device)

for _ in range(400):
    modelo_nn.train()
    optimizador.zero_grad()
    criterio(modelo_nn(X_t), y_t).backward()
    optimizador.step()

modelo_nn.eval()
with torch.no_grad():
    logits = modelo_nn(torch.tensor(X_te, dtype=torch.float32, device=device))
    pred_nn = (torch.sigmoid(logits) >= 0.5).cpu().numpy().astype(int)

acc_nn = accuracy_score(y_test, pred_nn)

predicciones_test["PyTorch_MLP"] = pred_nn



## 8. Comparación de todos los modelos (sklearn + PyTorch)

Misma partición **test**. Tabla por **accuracy**; debajo, reporte del **mejor** modelo.


In [7]:
comparacion = pd.concat(
    [
        pd.DataFrame(filas),
        pd.DataFrame([{"modelo": "PyTorch_MLP", "accuracy": acc_nn}]),
    ],
    ignore_index=True,
).sort_values("accuracy", ascending=False)

display(comparacion.round(4))

mejor_nombre = comparacion.iloc[0]["modelo"]
print(f"\nMejor modelo en test: {mejor_nombre} (accuracy = {comparacion.iloc[0]['accuracy']:.4f})")
print(
    classification_report(
        y_test,
        predicciones_test[mejor_nombre],
        target_names=["No spam (0)", "Spam (1)"],
    )
)



,modelo,accuracy
0,LogisticRegression,1.0
1,SGDClassifier,1.0
2,SVC,1.0
3,OneVsOneClassifier,1.0
4,OneVsRestClassifier,1.0
5,KNN,1.0
6,DecisionTree,1.0
7,RandomForest,1.0
8,GradientBoosting,1.0
12,PyTorch_MLP,1.0



Mejor modelo en test: LogisticRegression (accuracy = 1.0000)
              precision    recall  f1-score   support

 No spam (0)       1.00      1.00      1.00         1
    Spam (1)       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

